# Appendix C. Mismatches in OD trip distributions

This notebook computes the **OD-level mismatch indicators** used to compare the weighted HTS and averaged LCS matrices.

It follows the manuscript flow:

1. Load the OD-level HTS and LCS comparison tables  
2. Standardize table structure for consistent merging  
3. Compute **SRMSE** as the subgroup-level mismatch indicator for OD trip distributions  
4. Compute **zero-cell metrics** as supplementary indicators of OD coverage  
5. Export the final subgroup-level comparison tables

In this notebook, **LCS is treated as the reference dataset**, consistent with the manuscript. SRMSE is computed from **OD trip proportions within each subgroup across all OD pairs in the common OD frame, including zero-flow cells**. Zero-cell metrics are reported separately to describe the sparsity and asymmetry of observed OD coverage.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Base directories
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "2_data_OD"
RESULT_DIR = BASE_DIR / "3_Result_OD_level_Mismatch"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Input files
OD_HTS_FILE = DATA_DIR / "OD_HTS.csv"
OD_HTS_GENDER_FILE     = DATA_DIR / "OD_HTS_gender.csv"
OD_HTS_AGEGROUP_FILE   = DATA_DIR / "OD_HTS_agegroup.csv"
OD_HTS_TRAVELTYPE_FILE = DATA_DIR / "OD_HTS_traveltype.csv"
OD_HTS_TIMEZONE_FILE   = DATA_DIR / "OD_HTS_timezone.csv"
OD_HTS_TRAVELTYPE_TIMEZONE_FILE = DATA_DIR / "OD_HTS_traveltype_timezone.csv"

OD_LCS_FILE = DATA_DIR / "OD_LCS.csv"
OD_LCS_GENDER_FILE     = DATA_DIR / "OD_LCS_gender.csv"
OD_LCS_AGEGROUP_FILE   = DATA_DIR / "OD_LCS_agegroup.csv"
OD_LCS_TRAVELTYPE_FILE = DATA_DIR / "OD_LCS_traveltype.csv"
OD_LCS_TIMEZONE_FILE   = DATA_DIR / "OD_LCS_timezone.csv"
OD_LCS_TRAVELTYPE_TIMEZONE_FILE = DATA_DIR / "OD_LCS_traveltype_timezone.csv"

# Output files (Subgroup-level)
MISMATCH_GENDER_FILE = RESULT_DIR / "mismatch_gender.csv"
MISMATCH_AGEGROUP_FILE = RESULT_DIR / "mismatch_agegroup.csv"
MISMATCH_TRAVELTYPE_FILE = RESULT_DIR / "mismatch_traveltype.csv"
MISMATCH_TIMEZONE_FILE = RESULT_DIR / "mismatch_timezone.csv"
MISMATCH_TRAVELTYPE_TIMEZONE_FILE = RESULT_DIR / "mismatch_traveltype_timezone.csv"

# Origin- and destination-level SRMSE outputs
REGRESSION_DATA_DIR = BASE_DIR / "4_data_regression"
REGRESSION_DATA_DIR.mkdir(parents=True, exist_ok=True)

ORIGIN_SRMSE_FILE = (REGRESSION_DATA_DIR / "origin_srmse_overall.csv")
DESTINATION_SRMSE_FILE = (REGRESSION_DATA_DIR / "destination_srmse_overall.csv")

N_SEOUL_ZONES = 424

## 1. Load OD-level comparison tables

These tables were prepared in Appendix C(1) and Appendix C(2).  
Each table represents trips aggregated to the common Seoul OD system and aligned to the same subgroup definition.

In [ ]:
OD_HTS = pd.read_csv(OD_HTS_FILE)
OD_HTS_gender = pd.read_csv(OD_HTS_GENDER_FILE)
OD_HTS_agegroup = pd.read_csv(OD_HTS_AGEGROUP_FILE)
OD_HTS_traveltype = pd.read_csv(OD_HTS_TRAVELTYPE_FILE)
OD_HTS_timezone = pd.read_csv(OD_HTS_TIMEZONE_FILE)
OD_HTS_traveltype_timezone = pd.read_csv(OD_HTS_TRAVELTYPE_TIMEZONE_FILE)

OD_LCS = pd.read_csv(OD_LCS_FILE)
OD_LCS_gender = pd.read_csv(OD_LCS_GENDER_FILE)
OD_LCS_agegroup = pd.read_csv(OD_LCS_AGEGROUP_FILE)
OD_LCS_traveltype = pd.read_csv(OD_LCS_TRAVELTYPE_FILE)
OD_LCS_timezone = pd.read_csv(OD_LCS_TIMEZONE_FILE)
OD_LCS_traveltype_timezone = pd.read_csv(OD_LCS_TRAVELTYPE_TIMEZONE_FILE)

## 2. Define helper functions

The functions below standardize OD keys, merge HTS and LCS tables, and compute subgroup-level mismatch indicators.

Following the manuscript, **SRMSE** is defined using the **OD trip proportion** of each subgroup in HTS and LCS.
For each subgroup, the denominator is the **total number of trips within that subgroup**, so the resulting values represent the OD trip distribution of that subgroup itself.
**SRMSE is calculated across all 179,776 OD pairs in the common 424 × 424 OD frame, including zero-flow cells.**

In [ ]:
def standardize_od_table_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize key OD columns for stable merging across HTS and LCS tables.
    """
    out = df.copy()

    if "origin_id" in out.columns:
        out["origin_id"] = pd.to_numeric(out["origin_id"], errors="coerce").astype("Int64")
    if "destination_id" in out.columns:
        out["destination_id"] = pd.to_numeric(out["destination_id"], errors="coerce").astype("Int64")

    if "OD_pair" not in out.columns and {"origin_id", "destination_id"}.issubset(out.columns):
        out["OD_pair"] = (
            out["origin_id"].astype(str).str.strip()
            + "_"
            + out["destination_id"].astype(str).str.strip()
        )

    return out


def safe_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Return R-squared only when variation exists in the reference values.
    """
    if len(y_true) < 2:
        return np.nan
    if np.allclose(y_true, y_true[0]):
        return np.nan
    return r2_score(y_true, y_pred)

In [ ]:
def compute_metrics_long(
    hts_df: pd.DataFrame,
    lcs_df: pd.DataFrame,
    file_key: str,
    indicator_col: str,
) -> pd.DataFrame:
    """
    Compare weighted HTS and averaged LCS OD trip distributions
    for one subgroup definition.

    Input format:
    OD_pair, origin_id, destination_id, <indicator>, trips

    LCS is treated as the reference dataset.
    SRMSE is computed as RMSE divided by the mean of LCS subgroup-specific
    OD trip proportions (i.e., SRMSE_mean).
    Zero-cell metrics summarize differences in observed OD coverage.
    """
    hts_df = standardize_od_table_columns(hts_df)
    lcs_df = standardize_od_table_columns(lcs_df)

    required_cols = ["OD_pair", "origin_id", "destination_id", indicator_col, "trips"]

    hts = hts_df[required_cols].copy()
    lcs = lcs_df[required_cols].copy()

    hts["trips"] = pd.to_numeric(hts["trips"], errors="coerce").fillna(0)
    lcs["trips"] = pd.to_numeric(lcs["trips"], errors="coerce").fillna(0)

    hts = hts.rename(columns={"trips": "trips_hts"})
    lcs = lcs.rename(columns={"trips": "trips_lcs"})

    merged = pd.merge(
        hts,
        lcs,
        on=["OD_pair", "origin_id", "destination_id", indicator_col],
        how="outer",
    )

    merged["trips_hts"] = pd.to_numeric(merged["trips_hts"], errors="coerce").fillna(0)
    merged["trips_lcs"] = pd.to_numeric(merged["trips_lcs"], errors="coerce").fillna(0)

    results = []

    for level, group in merged.groupby(indicator_col, dropna=False, sort=True):
        group_local = group.copy()

        # Compute subgroup-specific OD trip proportions.
        total_hts_subgroup = group_local["trips_hts"].sum()
        total_lcs_subgroup = group_local["trips_lcs"].sum()

        group_local["prop_hts"] = np.where(
            total_hts_subgroup > 0,
            group_local["trips_hts"] / total_hts_subgroup,
            0.0,
        )
        group_local["prop_lcs"] = np.where(
            total_lcs_subgroup > 0,
            group_local["trips_lcs"] / total_lcs_subgroup,
            0.0,
        )

        y_pred = group_local["prop_hts"].to_numpy()
        y_true = group_local["prop_lcs"].to_numpy()

        rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
        srmse = rmse / (y_true.mean() + 1e-10) if len(y_true) > 0 else np.nan

        r2 = safe_r2(y_true, y_pred)

        # Zero-cell metrics are computed from trip counts, not proportions.
        hts_zero = group_local["trips_hts"].eq(0)
        lcs_zero = group_local["trips_lcs"].eq(0)
        lcs_nonzero = group_local["trips_lcs"].ne(0)

        total_cells = len(group_local)

        zero_hts = int(hts_zero.sum())
        zero_lcs = int(lcs_zero.sum())
        both_zero = int((hts_zero & lcs_zero).sum())

        zero_rate_hts = zero_hts / total_cells if total_cells > 0 else np.nan
        zero_rate_lcs = zero_lcs / total_cells if total_cells > 0 else np.nan

        relative_zero_rate = (
            ((hts_zero) & (lcs_nonzero)).sum() / (lcs_nonzero.sum() + 1e-10)
        )
        both_zero_rate = both_zero / total_cells if total_cells > 0 else np.nan

        results.append({
            "category": file_key,
            "variable": level,
            "n_cells": total_cells,
            "SRMSE": round(srmse, 4) if pd.notna(srmse) else np.nan,
            "R_squared": round(r2, 3) if pd.notna(r2) else np.nan,
            "Zero_HTS": zero_hts,
            "Zero_LCS": zero_lcs,
            "ZeroRate_HTS": round(zero_rate_hts, 4) if pd.notna(zero_rate_hts) else np.nan,
            "ZeroRate_LCS": round(zero_rate_lcs, 4) if pd.notna(zero_rate_lcs) else np.nan,
            "RelZeroRate": round(relative_zero_rate, 4),
            "BothZeroCount": both_zero,
            "BothZeroRate": round(both_zero_rate, 4) if pd.notna(both_zero_rate) else np.nan,
        })

    return pd.DataFrame(results)

In [ ]:
def compute_metrics_two_indicators(
    hts_df: pd.DataFrame,
    lcs_df: pd.DataFrame,
    file_key1: str,
    file_key2: str,
    indicator_cols: list[str],
) -> pd.DataFrame:
    """
    Compare weighted HTS and averaged LCS OD trip distributions
    for a two-indicator subgroup table.

    Input format:
    OD_pair, origin_id, destination_id, <indicator1>, <indicator2>, trips

    LCS is treated as the reference dataset.
    SRMSE is computed as RMSE divided by the mean of LCS subgroup-specific
    OD trip proportions (i.e., SRMSE_mean).
    """
    hts_df = standardize_od_table_columns(hts_df)
    lcs_df = standardize_od_table_columns(lcs_df)

    required_cols = ["OD_pair", "origin_id", "destination_id"] + indicator_cols + ["trips"]

    missing_hts = [col for col in required_cols if col not in hts_df.columns]
    missing_lcs = [col for col in required_cols if col not in lcs_df.columns]

    if missing_hts:
        raise KeyError(
            f"Missing columns in HTS table: {missing_hts}\n"
            f"Available columns: {hts_df.columns.tolist()}"
        )
    if missing_lcs:
        raise KeyError(
            f"Missing columns in LCS table: {missing_lcs}\n"
            f"Available columns: {lcs_df.columns.tolist()}"
        )

    hts = hts_df[required_cols].copy()
    lcs = lcs_df[required_cols].copy()

    hts["trips"] = pd.to_numeric(hts["trips"], errors="coerce").fillna(0)
    lcs["trips"] = pd.to_numeric(lcs["trips"], errors="coerce").fillna(0)

    hts = hts.rename(columns={"trips": "trips_hts"})
    lcs = lcs.rename(columns={"trips": "trips_lcs"})

    merge_keys = ["OD_pair", "origin_id", "destination_id"] + indicator_cols
    merged = pd.merge(hts, lcs, on=merge_keys, how="outer")

    merged["trips_hts"] = pd.to_numeric(merged["trips_hts"], errors="coerce").fillna(0)
    merged["trips_lcs"] = pd.to_numeric(merged["trips_lcs"], errors="coerce").fillna(0)

    results = []

    for levels, group in merged.groupby(indicator_cols, dropna=False, sort=True):
        if not isinstance(levels, tuple):
            levels = (levels,)

        level_dict = dict(zip(indicator_cols, levels))
        group_local = group.copy()

        # Compute OD trip proportions within each interaction subgroup.
        total_hts_subgroup = group_local["trips_hts"].sum()
        total_lcs_subgroup = group_local["trips_lcs"].sum()

        group_local["prop_hts"] = np.where(
            total_hts_subgroup > 0,
            group_local["trips_hts"] / total_hts_subgroup,
            0.0,
        )
        group_local["prop_lcs"] = np.where(
            total_lcs_subgroup > 0,
            group_local["trips_lcs"] / total_lcs_subgroup,
            0.0,
        )

        y_pred = group_local["prop_hts"].to_numpy()
        y_true = group_local["prop_lcs"].to_numpy()

        rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
        srmse = rmse / (y_true.mean() + 1e-10) if len(y_true) > 0 else np.nan

        r2 = safe_r2(y_true, y_pred)

        hts_zero = group_local["trips_hts"].eq(0)
        lcs_zero = group_local["trips_lcs"].eq(0)
        lcs_nonzero = group_local["trips_lcs"].ne(0)

        total_cells = len(group_local)

        zero_hts = int(hts_zero.sum())
        zero_lcs = int(lcs_zero.sum())
        both_zero = int((hts_zero & lcs_zero).sum())

        zero_rate_hts = zero_hts / total_cells if total_cells > 0 else np.nan
        zero_rate_lcs = zero_lcs / total_cells if total_cells > 0 else np.nan
        relative_zero_rate = ((hts_zero) & (lcs_nonzero)).sum() / (lcs_nonzero.sum() + 1e-10)
        both_zero_rate = both_zero / total_cells if total_cells > 0 else np.nan

        results.append({
            "category": f"{file_key1}_{file_key2}",
            indicator_cols[0]: level_dict[indicator_cols[0]],
            indicator_cols[1]: level_dict[indicator_cols[1]],
            "variable": f"{level_dict[indicator_cols[0]]}_{level_dict[indicator_cols[1]]}",
            "n_cells": total_cells,
            "SRMSE": round(srmse, 4) if pd.notna(srmse) else np.nan,
            "R_squared": round(r2, 3) if pd.notna(r2) else np.nan,
            "Zero_HTS": zero_hts,
            "Zero_LCS": zero_lcs,
            "ZeroRate_HTS": round(zero_rate_hts, 4) if pd.notna(zero_rate_hts) else np.nan,
            "ZeroRate_LCS": round(zero_rate_lcs, 4) if pd.notna(zero_rate_lcs) else np.nan,
            "RelZeroRate": round(relative_zero_rate, 4),
            "BothZeroCount": both_zero,
            "BothZeroRate": round(both_zero_rate, 4) if pd.notna(both_zero_rate) else np.nan,
        })

    return pd.DataFrame(results)

## 3. Construct Origin-SRMSE and Destination-SRMSE

- **Origin-SRMSE** compares the destination-trip distributions of HTS and LCS within each origin.
- **Destination-SRMSE** compares the origin-trip distributions of HTS and LCS within each destination.

In [ ]:
def standardize_zone_id(series: pd.Series) -> pd.Series:
    """Convert zone IDs to stable string keys."""
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    ).astype("Int64")

    if numeric.isna().any():
        raise ValueError(
            f"Zone ID contains {numeric.isna().sum():,} "
            "missing or invalid values."
        )

    return numeric.astype("string")


def standardize_od_table(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """Standardize OD table columns for stable merging."""
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()

    rename_map = {
        "출발행정동코드": "origin_id",
        "도착행정동코드": "destination_id",
        "통행횟수": "trips",
        "통행량": "trips",
    }
    out = out.rename(columns=rename_map)

    required = [
        "origin_id",
        "destination_id",
        "trips",
    ]

    missing = [
        col for col in required
        if col not in out.columns
    ]

    if missing:
        raise KeyError(
            f"Missing OD columns: {missing}. "
            f"Available columns: {out.columns.tolist()}"
        )

    out["origin_id"] = standardize_zone_id(
        out["origin_id"]
    )
    out["destination_id"] = standardize_zone_id(
        out["destination_id"]
    )

    trips = pd.to_numeric(
        out["trips"],
        errors="coerce",
    )

    if trips.isna().any():
        raise ValueError(
            f"Trip column contains "
            f"{trips.isna().sum():,} missing or invalid values."
        )

    if trips.lt(0).any():
        raise ValueError(
            "Trip column contains negative values."
        )

    out["trips"] = trips.astype(float)

    out["OD_pair"] = (
        out["origin_id"].astype(str)
        + "_"
        + out["destination_id"].astype(str)
    )

    return out


def compute_manuscript_srmse(
    hts_share: np.ndarray,
    lcs_share: np.ndarray,
) -> float:
    """Compute SRMSE as RMSE divided by the mean LCS share."""
    hts_share = np.asarray(
        hts_share,
        dtype=float,
    )
    lcs_share = np.asarray(
        lcs_share,
        dtype=float,
    )

    valid = (
        np.isfinite(hts_share)
        & np.isfinite(lcs_share)
    )

    if not valid.any():
        return np.nan

    y_pred = hts_share[valid]
    y_true = lcs_share[valid]

    rmse = np.sqrt(
        np.mean((y_pred - y_true) ** 2)
    )

    # Retained for numerical consistency with the reported analysis.
    return float(
        rmse / (np.mean(y_true) + 1e-10)
    )


def build_overall_spatial_srmse(
    hts_df: pd.DataFrame,
    lcs_df: pd.DataFrame,
    axis: str,
) -> pd.DataFrame:
    """Compute overall Origin-SRMSE or Destination-SRMSE."""
    hts_df = standardize_od_table(hts_df)
    lcs_df = standardize_od_table(lcs_df)

    key_cols = [
        "OD_pair",
        "origin_id",
        "destination_id",
    ]

    hts = (
        hts_df[key_cols + ["trips"]]
        .rename(columns={"trips": "trips_hts"})
        .copy()
    )

    lcs = (
        lcs_df[key_cols + ["trips"]]
        .rename(columns={"trips": "trips_lcs"})
        .copy()
    )

    merged = hts.merge(
        lcs,
        on=key_cols,
        how="outer",
    )

    merged["trips_hts"] = (
        merged["trips_hts"].fillna(0.0)
    )
    merged["trips_lcs"] = (
        merged["trips_lcs"].fillna(0.0)
    )

    if axis == "origin":
        spatial_col = "origin_id"
        counterpart_col = "destination_id"
        metric_col = "Origin_SRMSE"

    elif axis == "destination":
        spatial_col = "destination_id"
        counterpart_col = "origin_id"
        metric_col = "Destination_SRMSE"

    else:
        raise ValueError(
            "axis must be either 'origin' or 'destination'."
        )

    results = []

    for spatial_id, group in merged.groupby(
        spatial_col,
        sort=True,
        dropna=False,
    ):
        total_hts = group["trips_hts"].sum()
        total_lcs = group["trips_lcs"].sum()

        share_hts = np.where(
            total_hts > 0,
            group["trips_hts"] / total_hts,
            0.0,
        )

        share_lcs = np.where(
            total_lcs > 0,
            group["trips_lcs"] / total_lcs,
            0.0,
        )

        results.append({
            spatial_col: spatial_id,
            metric_col: compute_manuscript_srmse(
                share_hts,
                share_lcs,
            ),
            "HTS_total_trips_in_unit": float(total_hts),
            "LCS_total_trips_in_unit": float(total_lcs),
            "n_od_cells": int(len(group)),
            "n_positive_lcs_cells": int(
                (share_lcs > 0).sum()
            ),
            "n_unique_counterpart_zones": int(
                group[counterpart_col].nunique(
                    dropna=True
                )
            ),
        })

    result = (
        pd.DataFrame(results)
        .sort_values(spatial_col)
        .reset_index(drop=True)
    )

    if len(result) != N_SEOUL_ZONES:
        raise ValueError(
            f"Expected {N_SEOUL_ZONES} spatial units, "
            f"but found {len(result)}."
        )

    if not result["n_od_cells"].eq(
        N_SEOUL_ZONES
    ).all():
        raise ValueError(
            "Each spatial unit must contain exactly "
            f"{N_SEOUL_ZONES} counterpart OD cells."
        )

    return result

In [ ]:
ORIGIN_SRMSE = build_overall_spatial_srmse(
    hts_df=OD_HTS,
    lcs_df=OD_LCS,
    axis="origin",
)

DESTINATION_SRMSE = build_overall_spatial_srmse(
    hts_df=OD_HTS,
    lcs_df=OD_LCS,
    axis="destination",
)

ORIGIN_SRMSE.to_csv(
    ORIGIN_SRMSE_FILE,
    index=False,
    encoding="utf-8-sig",
)

DESTINATION_SRMSE.to_csv(
    DESTINATION_SRMSE_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("Save complete")
# print(f"Saved: {ORIGIN_SRMSE_FILE}")
# print(f"Saved: {DESTINATION_SRMSE_FILE}")

display(ORIGIN_SRMSE.head())
display(DESTINATION_SRMSE.head())

## 4. Compute subgroup-level mismatch indicators

This step computes the mismatch indicators reported in the manuscript for:
- gender,
- age group,
- travel type,
- time zone, and
- travel type × time zone.

SRMSE captures the **mismatch in OD trip distributions** within each subgroup.  
The zero-cell metrics are reported as **supplementary indicators** describing the sparsity and asymmetry of observed OD coverage. 

In [ ]:
MISMATCH_GENDER = compute_metrics_long(
    hts_df=OD_HTS_gender,
    lcs_df=OD_LCS_gender,
    file_key="gender",
    indicator_col="gender",
)

MISMATCH_AGEGROUP = compute_metrics_long(
    hts_df=OD_HTS_agegroup,
    lcs_df=OD_LCS_agegroup,
    file_key="age_group",
    indicator_col="age_group",
)

MISMATCH_TRAVELTYPE = compute_metrics_long(
    hts_df=OD_HTS_traveltype,
    lcs_df=OD_LCS_traveltype,
    file_key="travel_type",
    indicator_col="travel_type",
)

MISMATCH_TIMEZONE = compute_metrics_long(
    hts_df=OD_HTS_timezone,
    lcs_df=OD_LCS_timezone,
    file_key="time_zone",
    indicator_col="time_zone",
)

MISMATCH_TRAVELTYPE_TIMEZONE = compute_metrics_two_indicators(
    hts_df=OD_HTS_traveltype_timezone,
    lcs_df=OD_LCS_traveltype_timezone,
    file_key1="travel_type",
    file_key2="time_zone",
    indicator_cols=["travel_type", "time_zone"],
)

### (Additional Analysis) Mismatches by Travel Type × Time Zone

This section computes OD-level mismatch indicators for the interaction
between travel type and time zone, corresponding to the additional
appendix analysis reported in the manuscript.

In [ ]:
cols = ["travel_type","time_zone","variable","SRMSE","RelZeroRate"]
MISMATCH_TRAVELTYPE_TIMEZONE[cols]

## 5. Save

In [ ]:
MISMATCH_GENDER.to_csv(MISMATCH_GENDER_FILE, index=False, encoding="utf-8-sig")
MISMATCH_AGEGROUP.to_csv(MISMATCH_AGEGROUP_FILE, index=False, encoding="utf-8-sig")
MISMATCH_TRAVELTYPE.to_csv(MISMATCH_TRAVELTYPE_FILE, index=False, encoding="utf-8-sig")
MISMATCH_TIMEZONE.to_csv(MISMATCH_TIMEZONE_FILE, index=False, encoding="utf-8-sig")
MISMATCH_TRAVELTYPE_TIMEZONE.to_csv(MISMATCH_TRAVELTYPE_TIMEZONE_FILE, index=False, encoding="utf-8-sig")

print('Save complete')
# print(f"Saved: {MISMATCH_GENDER_FILE}")
# print(f"Saved: {MISMATCH_AGEGROUP_FILE}")
# print(f"Saved: {MISMATCH_TRAVELTYPE_FILE}")
# print(f"Saved: {MISMATCH_TIMEZONE_FILE}")
# print(f"Saved: {MISMATCH_TRAVELTYPE_TIMEZONE_FILE}")